# Experiment 4: Hill Climbing and Random Restarts
**Course:** CSAI4014 – Algorithms for Intelligent Systems and Robotics

Run all cells from top to bottom. The experiment plots a multimodal function, applies steepest-ascent hill climbing to the eight-queens problem by minimising conflicts, and compares measured restart success rates with the independent-trial prediction.

## 1. Aim
To study local search, demonstrate how hill climbing can become trapped at local maxima, and evaluate random restarts on the eight-queens problem.

## 2. Objectives
1. Plot a one-dimensional function with several local maxima.
2. Implement steepest-ascent hill climbing for eight queens using the number of conflicts as the objective.
3. Measure success over at least 200 random starts.
4. Implement random restarts and compare measured rates with 1 − (1 − p)^k.

## 3. Requirements
Python, NumPy and Matplotlib. No hardware is required.

## 4. Theory
Hill climbing starts from a complete candidate and repeatedly moves to a neighbouring candidate with a better objective value. It uses little memory but can stop at a local maximum/minimum, plateau or ridge. For eight queens, minimise the number of attacking pairs; zero conflicts is a solution. If one independent run succeeds with probability p, k restarts succeed with probability 1 − (1 − p)^k.

## 5. Example 1 — Seeing the landscape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(0, 10, 600)
fx = np.sin(xs) + 0.6*np.sin(2.7*xs) + 0.35*np.sin(5.1*xs) - 0.05*(xs-5)**2

plt.figure(figsize=(9, 3))
plt.plot(xs, fx, lw=1.1)
plt.plot(xs[np.argmax(fx)], fx.max(), 'o', mfc='none')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Function with multiple local maxima')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Example 2 — Hill climbing on eight queens

In [ ]:
import random

def conflicts(board):
    n = len(board)
    bad = 0
    for i in range(n):
        for j in range(i+1, n):
            if board[i] == board[j] or abs(board[i]-board[j]) == abs(i-j):
                bad += 1
    return bad

def hill_climb(n=8, seed=0):
    r = random.Random(seed)
    board = [r.randrange(n) for _ in range(n)]
    while True:
        current = conflicts(board)
        if current == 0:
            return True
        best, move = current, None
        for c in range(n):
            old = board[c]
            for row in range(n):
                if row == old:
                    continue
                board[c] = row
                v = conflicts(board)
                if v < best:
                    best, move = v, (c, row)
            board[c] = old
        if move is None:
            return False
        board[move[0]] = move[1]

# One run for a reproducible demonstration
print('Solved:', hill_climb(8, seed=0))


## 7. Example 3 — Random restarts and 200 trials

In [ ]:
def hill_climb_restarts(n=8, restarts=10, seed=0):
    for k in range(restarts):
        if hill_climb(n, seed=seed*1000 + k):
            return True
    return False

trials = 200
plain = sum(hill_climb(8, seed=s) for s in range(trials)) / trials
print(f'Plain hill climbing: {100*plain:.1f}% of {trials} random starts')

restart_counts = (1, 2, 5, 10, 20)
measured = []
for k in restart_counts:
    ok = sum(hill_climb_restarts(8, k, seed=1000+s)
             for s in range(trials)) / trials
    measured.append(ok)
    print(f'{k:>3} restarts: {100*ok:5.1f}% of {trials} trials')

predicted = [1 - (1-plain)**k for k in restart_counts]
print('Predicted rates:', [round(100*x, 1) for x in predicted])


## 8. Plot measured and predicted success rates

In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.plot(restart_counts, np.array(measured)*100, 'o-', label='Measured')
plt.plot(restart_counts, np.array(predicted)*100, 's--',
         label=f'Predicted: 1-(1-p)^k, p={plain:.3f}')
plt.xlabel('Number of random restarts')
plt.ylabel('Success rate (%)')
plt.title('Eight-queens hill climbing')
plt.xticks(restart_counts)
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 9. Results and observations
Record the plain hill-climbing success rate from 200 starts, and the measured success rate for 1, 2, 5, 10 and 20 restarts. State the number of trials beside every percentage. Compare measured values with 1 − (1 − p)^k, using the plain-run rate as p.

## 10. Discussion
A single hill-climbing run may stop at a local minimum because no single-queen move reduces the conflict count. Random restarts provide fresh initial boards and improve the probability of finding a solution. The measured percentages need not exactly match the prediction because each estimate uses a finite number of trials. Different seeded runs can produce slightly different percentages.

## 11. Applications
Local search with restarts is used for large scheduling and layout problems where systematic search is impractical. Similar approaches are used in robot motion planning to escape potential-field local minima and in machine learning through multiple initialisations.

## 12. Conclusion
The function plot demonstrated the presence of local maxima. Steepest-ascent hill climbing was implemented for eight queens by minimising conflicts. Experiments over 200 trials showed how random restarts improve the success rate, which was compared with the theoretical independent-run prediction.

## 13. Viva questions
1. State the three ways hill climbing gets stuck, and say which of them sideways moves can fix.
2. Why does hill climbing use constant memory when backtracking does not?
3. Your success rate is 15.5%. How many restarts are needed for 99% success, and what assumption does your calculation make?
4. Hill climbing starts with a complete board; backtracking builds one column by column. Which is better for n = 1000, and why?
5. You report 12.5% on one run and 15.5% on another. Is your program wrong? Justify.

## Source
CSAI4014 Algorithms for Intelligent Systems and Robotics Laboratory Manual, Experiment 4: Hill climbing, Chapter 5, pages 24–28.